# Smoke Test Data (~1 min run)

Creates a minimal dataset from `micro_test.chain` (3 chrX-chrX chains) for fast pipeline validation.

Selects human genes in the locus covered by those chains:
- **All protein-coding** genes
- **10–20 lncRNAs** (XIST guaranteed)
- **25–30 other ncRNAs** (snoRNA, miRNA, tRNA, etc.)

Uses the full hg38/mm39 2bit files (no subsetting needed).

In [ ]:
!chainFilter chains/hg38.mm39.allfilled.chain.gz -q=chrX -t=chrX -minScore=163685769 > chains/micro_test.chain

In [ ]:
from pathlib import Path
import pyrion
from pyrion.io.chain import read_chain_file
from pyrion.io.bed import read_bed12_file
from pyrion.io.gene_data import read_gene_data
from pyrion.core.genes import TranscriptsCollection
from pyrion.core.intervals import GenomicInterval
from pyrion.ops.chains import get_chain_t_start, get_chain_t_end

In [3]:
# Input paths
chain_file = Path("./chains/micro_test.chain")
ref_bed12 = Path("./reference_annotation/hg38.input.w.tRNA.bed")
biomart_tsv = Path("./reference_annotation/hg38.transcript_metadata.tsv")

# Output paths
out_chain_gz = Path("./chains/smoke_test.chain.gz")
out_bed = Path("./reference_annotation/smoke_test.bed")
out_metadata = Path("./reference_annotation/smoke_test.metadata.tsv")

In [4]:
chains = read_chain_file(str(chain_file))
print(chains.summary())

t_start = min(get_chain_t_start(a) for a in chains)
t_end = max(get_chain_t_end(a) for a in chains)
t_chrom = chains[0].t_chrom

covered_interval = GenomicInterval(t_chrom, t_start, t_end)
print(f"\nTarget locus covered: {covered_interval}")

3 genome alignments from chains/micro_test.chain

Target locus covered: chrX:10447550-155264946


In [5]:
def strip_version(tid: str) -> str:
    return tid.split(".")[0]

all_transcripts = read_bed12_file(str(ref_bed12))
all_transcripts = all_transcripts.copy_with_remapped_ids(strip_version)
print(f"Total transcripts: {len(all_transcripts)}")

gene_data = read_gene_data(
    str(biomart_tsv),
    transcript_id_column="transcript_id",
    gene_column="gene_id",
    transcript_type_column="transcript_biotype",
    separator="\t",
)
# strip versions from gene_data IDs
gene_data._transcript_to_gene = {strip_version(k): strip_version(v) for k, v in gene_data._transcript_to_gene.items()}
gene_data._gene_to_transcripts = {}
for tid, gid in gene_data._transcript_to_gene.items():
    gene_data._gene_to_transcripts.setdefault(gid, set()).add(tid)
gene_data._transcript_to_biotype = {strip_version(k): v for k, v in gene_data._transcript_to_biotype.items()}

all_transcripts.bind_gene_data(gene_data)
print(f"Gene data: {gene_data.summary()}")

Total transcripts: 386300
Gene data: GeneData: gene-transcript: 386300 pairs, transcript-biotype: 386300 pairs from reference_annotation/hg38.transcript_metadata.tsv


In [6]:
locus_transcripts = all_transcripts.get_transcripts_in_interval(covered_interval)
print(f"Transcripts in the covered locus: {len(locus_transcripts)}")

protein_coding = locus_transcripts.filter_by_biotype("protein_coding")
lncrna = locus_transcripts.filter_by_biotype("lncRNA")
snorna = locus_transcripts.filter_by_biotype("snoRNA")
mirna = locus_transcripts.filter_by_biotype("miRNA")
trna = locus_transcripts.filter_by_biotype("tRNA")

print(f"  protein_coding: {len(protein_coding)} transcripts ({len(protein_coding.genes)} genes)")
print(f"  lncRNA:         {len(lncrna)} transcripts ({len(lncrna.genes)} genes)")
print(f"  snoRNA:         {len(snorna)} transcripts ({len(snorna.genes)} genes)")
print(f"  miRNA:          {len(mirna)} transcripts ({len(mirna.genes)} genes)")
print(f"  tRNA:           {len(trna)} transcripts ({len(trna.genes)} genes)")

Transcripts in the covered locus: 10522
  protein_coding: 3186 transcripts (810 genes)
  lncRNA:         3905 transcripts (758 genes)
  snoRNA:         43 transcripts (43 genes)
  miRNA:          111 transcripts (111 genes)
  tRNA:           4 transcripts (4 genes)


In [8]:
import random
random.seed(42)

# All protein-coding: take canonical transcript per gene
protein_coding.canonize_transcripts()
pc_canonical = protein_coding.get_canonical_transcripts()
print(f"Protein-coding canonical transcripts: {len(pc_canonical)}")

# lncRNAs: pick 15 genes, XIST must be included
XIST_GENE_ID = "ENSG00000229807"
lncrna_genes = lncrna.genes
lncrna_gene_ids = [g.gene_id for g in lncrna_genes]

lncrna_selected_ids = set()
if XIST_GENE_ID in lncrna_gene_ids:
    lncrna_selected_ids.add(XIST_GENE_ID)

remaining = [gid for gid in lncrna_gene_ids if gid not in lncrna_selected_ids]
random.shuffle(remaining)
lncrna_selected_ids.update(remaining[:19])  # total 20
print(f"Selected lncRNA genes: {len(lncrna_selected_ids)}")

lncrna.canonize_transcripts()
lncrna_selected = TranscriptsCollection([
    gene.canonical_transcript for gene in lncrna_genes
    if gene.gene_id in lncrna_selected_ids and gene.has_canonical_transcript
])
print(f"lncRNA canonical transcripts: {len(lncrna_selected)}")

# Other ncRNAs: collect all snoRNA, miRNA, tRNA genes, pick ~28 total
other_ncrna_genes = []
for coll in [snorna, mirna, trna]:
    coll.canonize_transcripts()
    other_ncrna_genes.extend(coll.genes)

random.shuffle(other_ncrna_genes)
n_other = min(40, len(other_ncrna_genes))
other_selected_genes = other_ncrna_genes[:n_other]
other_ncrna_selected = TranscriptsCollection([
    g.canonical_transcript for g in other_selected_genes if g.has_canonical_transcript
])
print(f"Other ncRNA canonical transcripts: {len(other_ncrna_selected)}")

# Verify XIST
xist_check = [g for g in lncrna_genes if g.gene_id == XIST_GENE_ID]
if xist_check:
    print(f"\nXIST: {xist_check[0]}")
else:
    print("\nWARNING: XIST not found in locus!")

Protein-coding canonical transcripts: 810
Selected lncRNA genes: 20
lncRNA canonical transcripts: 20
Other ncRNA canonical transcripts: 40

XIST: Gene(id=ENSG00000229807, chrX:73820648-73852714:-1, 33 transcripts, canonical=ENST00000429829)


In [9]:
# Combine all selected transcripts
combined = TranscriptsCollection()
combined.extend(pc_canonical)
combined.extend(lncrna_selected)
combined.extend(other_ncrna_selected)

combined.bind_gene_data(gene_data)
combined.sort()
print(combined.summary())

# Show biotype breakdown
from collections import Counter
biotypes = Counter()
for t in combined:
    bt = gene_data.get_transcript_biotype(t.id)
    biotypes[bt] += 1
for bt, count in biotypes.most_common():
    print(f"  {bt}: {count}")

870 transcripts, 870 genes
  protein_coding: 810
  miRNA: 29
  lncRNA: 20
  snoRNA: 10
  tRNA: 1


In [10]:
# Save BED12
combined.save_to_bed12(str(out_bed))
print(f"Saved {len(combined)} transcripts → {out_bed}")

# Save metadata (trimmed to only selected transcripts)
combined.save_biodata(str(out_metadata))
print(f"Saved metadata → {out_metadata}")

Saved 870 transcripts → reference_annotation/smoke_test.bed
Saved metadata → reference_annotation/smoke_test.metadata.tsv


In [11]:
import gzip
import shutil

with open(str(chain_file), "rb") as f_in:
    with gzip.open(str(out_chain_gz), "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
print(f"Gzipped chain → {out_chain_gz}")

Gzipped chain → chains/smoke_test.chain.gz


In [12]:
# Verify outputs
smoke_bed = read_bed12_file(str(out_bed))
print(f"Smoke test BED: {len(smoke_bed)} transcripts")

smoke_chains = read_chain_file(str(out_chain_gz))
print(f"Smoke test chains: {len(smoke_chains)} chains")

import os
print(f"\nChain file size: {os.path.getsize(out_chain_gz) / 1024:.1f} KB")
print(f"BED file size:   {os.path.getsize(out_bed) / 1024:.1f} KB")

Smoke test BED: 870 transcripts
Smoke test chains: 3 chains

Chain file size: 2479.2 KB
BED file size:   124.8 KB


## Run command

From the project root:

```bash
./curia.py \
--ref-bed12 input_data/reference_annotation/smoke_test.bed \
--biomart-tsv input_data/reference_annotation/smoke_test.metadata.tsv \
--chain input_data/chains/smoke_test.chain.gz \
--ref-2bit input_data/2bit/hg38.test.subset.2bit \
--query-2bit input_data/2bit/mm39.test.subset.2bit \
--output-dir smoke_test_output \
--gpu-logger \
--cpu-max-workers 12 \
--gpu-min-batch 4 \
--gpu-max-batch 16
```